# Tugas Visualisasi Data dengan Matplotlib

Notebook ini membuat empat visualisasi dari data transaksi Point of Sale (POS) 2024-2025 dalam **satu gambar berukuran 2x2**. Jalankan sel dari atas ke bawah. Dataset dibaca dari folder utama repository. Datanya sama dengan tugas Manipulasi Data Pandas: `paket_a_transaksi_pos_2024_2025.csv`. Semua grafik hanya untuk **tahun 2025**.

| Posisi | Bagian | Visualisasi | Jenis grafik |
|---|---|---|---|
| Kiri atas | 1 | Penjualan per bulan tahun 2025 | Garis |
| Kanan atas | 2 | 10 produk terlaris tahun 2025 | Batang mendatar |
| Kiri bawah | 3 | Proporsi transaksi berhasil vs gagal/batal tahun 2025 | Pie |
| Kanan bawah | 4 | Hubungan jumlah beli dengan diskon | Sebar (*scatter*) |


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TAHUN = 2025
SEED = 42
FILE_DATA = Path("../paket_a_transaksi_pos_2024_2025.csv")
Path("hasil").mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

# Nama bulan dan warna yang dipakai di semua grafik
BULAN = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]
BIRU, HIJAU, MERAH, KUNING = "#3D85C6", "#81B29A", "#E07A5F", "#F2CC8F"

plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

#### Catatan sel: persiapan awal

Sel ini tidak menghasilkan angka. Fungsinya menyiapkan "meja kerja" sebelum mulai:

- **Tiga library utama:** `pandas` (tabel), `numpy` (hitungan dan angka acak), `matplotlib` (grafik).
- **`TAHUN` dan `FILE_DATA`:** nilai yang cukup diubah di sini jika suatu saat ingin menganalisis tahun lain atau memakai file di folder berbeda.
- **`rng` dengan `SEED = 42`:** pembuat angka acak yang hasilnya selalu sama. Dipakai untuk menggeser titik di Bagian 4, sehingga gambar tidak berubah tiap kali notebook dijalankan ulang.
- **Warna dan gaya grafik** ditetapkan sekali di sini agar keempat grafik tampil seragam.
- **Folder `hasil/`** dibuat otomatis untuk menyimpan gambar.

In [ ]:
df = pd.read_csv(FILE_DATA, dtype=str)      # semua kolom dibaca sebagai teks dulu
print("Jumlah baris awal      :", len(df))

# Hapus baris yang sama persis (duplikat)
df = df.drop_duplicates()
print("Setelah hapus duplikat :", len(df))
df.head()

#### Catatan sel: membaca data

Data dimuat ke tabel `df`, lalu baris kembar dibuang.

- **Dibaca sebagai teks (`dtype=str`).** Kolom tanggal dan harga memakai beberapa format. Jika pandas dibiarkan menebak, nilai seperti `13.000` bisa terbaca keliru, jadi konversinya dikerjakan sendiri di sel berikutnya.
- **`drop_duplicates()`.** Transaksi yang tercatat dua kali akan membuat penjualan terhitung ganda.
- **Cek di output:** 5.030 baris awal menjadi 5.000 baris, artinya ada 30 baris duplikat.

In [ ]:
# 1. Harga: ada yang ditulis "13.000" (titik ribuan) dan "7500,00" (koma desimal)
def bersihkan_harga(teks):
    teks = teks.strip().split(",")[0]       # "7500,00" menjadi "7500"
    return int(teks.replace(".", ""))       # "13.000"  menjadi "13000"

df["harga_satuan"] = df["harga_satuan"].apply(bersihkan_harga)


# 2. Tanggal: ada tiga format, semuanya diubah jadi tanggal asli
def bersihkan_tanggal(teks):
    teks = teks.strip()
    if teks[4] == "-":                                              # 2025-06-02
        return pd.to_datetime(teks, format="%Y-%m-%d")
    return pd.to_datetime(teks.replace("/", "-"), format="%d-%m-%Y")  # 26/03/2025 atau 23-12-2024

df["tanggal"] = df["tanggal"].apply(bersihkan_tanggal)

# 3. Jumlah dan diskon diubah menjadi angka
df["jumlah"] = df["jumlah"].astype(int)
df["diskon_persen"] = df["diskon_persen"].astype(int)

# 4. Jumlah negatif (retur/koreksi) dijadikan positif, seperti tugas Pandas
print("Jumlah negatif              :", (df["jumlah"] < 0).sum())
df["jumlah"] = df["jumlah"].abs()

# 5. Kode kasir yang kosong diisi K_UNKNOWN, seperti tugas Pandas
df["kode_kasir"] = df["kode_kasir"].fillna("K_UNKNOWN")

# 6. Jumlah di atas 12 dianggap salah input, jadi dibuang
print("Jumlah dibuang              :", (df["jumlah"] > 12).sum())
df = df[df["jumlah"] <= 12].copy()

# 7. Hitung Total_Bersih: (harga x jumlah) dikurangi diskon, lalu ditambah PPN 11%
df["Total_Bersih"] = df["harga_satuan"] * df["jumlah"] * (1 - df["diskon_persen"] / 100) * 1.11

print("Baris yang tersisa          :", len(df))

#### Catatan sel: membersihkan data

Tujuh perbaikan dilakukan berurutan. Tabel berikut merangkum masalah dan caranya:

| Yang diperbaiki | Masalah di data | Cara memperbaikinya |
|---|---|---|
| Harga | `13.000` memakai titik ribuan, `7500,00` memakai koma desimal | Buang bagian setelah koma dan semua titik, ubah ke angka bulat |
| Tanggal | Tiga format: `2025-06-02`, `26/03/2025`, `23-12-2024` | Dikenali per format, format ditulis eksplisit agar `05/03/2025` terbaca 5 Maret, bukan 3 Mei |
| Jumlah dan diskon | Masih berupa teks | Diubah menjadi bilangan bulat |
| Jumlah negatif | Retur atau koreksi tercatat minus | `.abs()` agar menjadi positif |
| Kode kasir kosong | Nilai `NaN` | Diisi `K_UNKNOWN`. Baris dipertahankan karena kolom ini tidak memengaruhi penjualan |
| Jumlah di atas 12 | Nilai ekstrem, misalnya ribuan unit | Dibuang karena jelas salah input dan merusak skala grafik |
| `Total_Bersih` | Belum ada | Harga x jumlah x (1 - diskon/100) x 1,11 (PPN 11% dihitung setelah diskon) |

**Cek di output:** 40 jumlah negatif diperbaiki, 12 baris di atas 12 dibuang, tersisa 4.988 baris.

In [ ]:
d25 = df[df["tanggal"].dt.year == TAHUN]
selesai = d25[d25["status"] == "SELESAI"]

print("Transaksi 2025 :", len(d25))
print("Yang SELESAI   :", len(selesai))
print("Jumlah min     :", d25["jumlah"].min(),)
print("Jumlah max     :", d25["jumlah"].max())
print("Nilai kosong   :", d25.isna().sum().sum())

#### Catatan sel: memisahkan data 2025

Dari data yang sudah bersih, dibuat dua tabel yang dipakai bergantian:

| Tabel | Isi | Dipakai untuk |
|---|---|---|
| `d25` | Semua transaksi 2025, apa pun statusnya | Bagian 3 (proporsi status) |
| `selesai` | Transaksi 2025 berstatus `SELESAI` | Bagian 1, 2, dan 4 |

Bagian 3 butuh semua status karena yang dihitung memang perbandingan antarstatus. Bagian lain hanya memakai transaksi yang benar-benar terjadi.

**Cek di output:** 2.502 transaksi di 2025, 2.365 di antaranya `SELESAI`. Jumlah berada di rentang 1-12 dan tidak ada nilai kosong tersisa.

---
## Bagian 1: Penjualan per Bulan Tahun 2025 (kiri atas)

**Pertanyaan:** bagaimana naik turunnya penjualan sepanjang 2025, dan bulan apa yang tertinggi dan terendah?

**Jenis grafik: garis.** Datanya berurutan menurut waktu, sehingga garis menunjukkan tren dengan jelas.

In [ ]:
# Total penjualan tiap bulan, dalam juta Rupiah
jual_bulan = (selesai.groupby(selesai["tanggal"].dt.month)["Total_Bersih"].sum()
              .reindex(range(1, 13), fill_value=0))
jual_juta = jual_bulan / 1_000_000

b_max, b_min = jual_juta.idxmax(), jual_juta.idxmin()

print("Total setahun : Rp", f"{jual_bulan.sum():,.0f}".replace(",", "."))
print("Rata-rata     :", round(jual_juta.mean(), 1), "juta per bulan")
print("Tertinggi     :", BULAN[b_max - 1], round(jual_juta[b_max], 1), "juta")
print("Terendah      :", BULAN[b_min - 1], round(jual_juta[b_min], 1), "juta")

#### Catatan sel: hitungan Bagian 1

1. `groupby` mengelompokkan transaksi `SELESAI` menurut bulan, lalu `.sum()` menjumlahkan `Total_Bersih` tiap bulan.
2. `reindex(range(1, 13))` memastikan ke-12 bulan selalu muncul. Bulan tanpa transaksi akan terisi 0, bukan hilang dari grafik.
3. Hasilnya dibagi 1.000.000 supaya sumbu grafik berisi angka kecil (juta Rupiah).
4. `idxmax()` dan `idxmin()` mencari nomor bulan tertinggi dan terendah, untuk diberi warna khusus di grafik.

**Cek di output:** total Rp 151.478.786, tertinggi di Maret, terendah di April.

---
## Bagian 2: 10 Produk Terlaris Tahun 2025 (kanan atas)


In [ ]:
# Unit terjual dan omzet tiap produk, lalu ambil 10 terbesar menurut unit
produk = (selesai.groupby("kode_produk")
          .agg(unit=("jumlah", "sum"), omzet=("Total_Bersih", "sum"))
          .sort_values(by=["unit", "omzet"], ascending=False)    # jika unit seri, omzet yang menentukan
          .head(10))
produk["omzet_juta"] = produk["omzet"] / 1_000_000

# Cek seri di batas peringkat 10
semua_unit = selesai.groupby("kode_produk")["jumlah"].sum().sort_values(ascending=False)
print("Jumlah produk yang terjual:", len(semua_unit))
print("Unit peringkat 10 =", semua_unit.iloc[9], "| peringkat 11 =", semua_unit.iloc[10])

produk[["unit", "omzet_juta"]].round(1)

#### Catatan sel: hitungan Bagian 2

Unit dan omzet dijumlahkan per `kode_produk`. Peringkat dibuat dari **unit**, karena "terlaris" berarti paling banyak laku. Jika dua produk seri di unit, produk dengan omzet lebih besar didahulukan sehingga urutannya selalu pasti. Omzet ikut dihitung karena unit saja bisa menyesatkan: produk murah bisa laku banyak tetapi sumbangan uangnya kecil.

**Cek seri di batas ke-10:** peringkat 10 memiliki 118 unit, sedangkan peringkat 11 hanya 111 unit. Jadi 10 besar ini pasti, tidak ada produk yang tersingkir karena seri.

---
## Bagian 3: Proporsi Transaksi Berhasil vs Gagal/Batal Tahun 2025 (kiri bawah)

**Pertanyaan:** dari seluruh transaksi 2025, berapa persen yang berhasil dan berapa persen yang gagal?

**Jenis grafik: pie.** Yang dilihat adalah porsi dari keseluruhan, dan kategorinya hanya tiga. Berhasil = `SELESAI`. Gagal/batal = `BATAL` + `RETUR`.

In [ ]:
# Jumlah transaksi untuk tiap status (semua status, jadi memakai d25)
status = d25["status"].value_counts().reindex(["SELESAI", "BATAL", "RETUR"])
persen = status / status.sum() * 100

gagal = status["BATAL"] + status["RETUR"]
persen_gagal = gagal / status.sum() * 100

print(status)
print(persen.round(1))
print(f"Berhasil     : {status['SELESAI']} ({persen['SELESAI']:.1f}%)")
print(f"Gagal/batal  : {gagal} ({persen_gagal:.1f}%)  = BATAL {status['BATAL']} + RETUR {status['RETUR']}")

#### Catatan sel: hitungan Bagian 3

Pengertian tiga status:
- **SELESAI:** transaksi berhasil.
- **BATAL:** transaksi tidak jadi.
- **RETUR:** barang sudah dibeli, lalu dikembalikan.

BATAL dan RETUR tetap dipisah di grafik supaya terlihat mana yang lebih sering terjadi, sedangkan gabungan keduanya dihitung sebagai transaksi gagal.


---
## Bagian 4: Hubungan Jumlah Beli dengan Diskon (kanan bawah)

**Pertanyaan:** apakah diskon yang lebih besar membuat orang membeli lebih banyak?

**Jenis grafik: sebar (*scatter*)** dengan garis rata-rata. Dipakai transaksi `SELESAI` tahun 2025.

In [ ]:
diskon = selesai["diskon_persen"]
jumlah = selesai["jumlah"]

diskon_geser = diskon + rng.uniform(-1, 1, len(selesai))
jumlah_geser = jumlah + rng.uniform(-0.25, 0.25, len(selesai))

ringkas = selesai.groupby("diskon_persen")["jumlah"].agg(
    n="count", rata_rata="mean", median="median", std="std")
ringkas["ci95"] = 1.96 * ringkas["std"] / np.sqrt(ringkas["n"])    # rentang wajar rata-rata


print("Korelasi Pearson (r)   :", round(r_pearson, 3))
print("Korelasi Spearman (rho):", round(rho, 3))
print("Transaksi yang memakai diskon:", round((diskon > 0).mean() * 100, 1), "%")

#### Catatan sel: hitungan Bagian 4

**1. Geseran titik (*jitter*).** Diskon hanya bernilai 0, 5, 10, 15, atau 20%, dan jumlah beli selalu bilangan bulat. Tanpa digeser, ribuan titik akan menumpuk di tempat yang sama. Geseran acak kecil hanya mengubah tampilan, bukan data yang dihitung.

**2. Ringkasan per tingkat diskon.** `n` adalah banyak transaksi, `rata_rata` dan `median` adalah jumlah beli yang umum, dan `ci95` adalah rentang wajar rata-rata. Semakin sedikit datanya, semakin lebar rentangnya.

**3. Dua ukuran korelasi.** Keduanya berkisar dari -1 sampai 1, dan nilai 0 berarti tidak ada hubungan. **Pearson** mengukur hubungan lurus. **Spearman** mengukur hubungan berdasarkan peringkat, dan lebih cocok di sini karena kedua kolom hanya punya beberapa nilai tetap. Jika keduanya sama-sama mendekati 0, kesimpulannya lebih kuat.

In [ ]:
# Satu gambar berisi 4 grafik (2 baris x 2 kolom)
fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)


def koma(angka, desimal=1):
    """Ubah 12.3 menjadi '12,3' (format Indonesia)."""
    return f"{angka:.{desimal}f}".replace(".", ",")


# Kiri atas: Bagian 1, penjualan per bulan
ax = axes[0, 0]
ax.plot(BULAN, jual_juta.values, color=BIRU, marker="o", linewidth=2)
ax.scatter(BULAN[b_max - 1], jual_juta[b_max], color=HIJAU, s=130, zorder=3, label="Tertinggi")
ax.scatter(BULAN[b_min - 1], jual_juta[b_min], color=MERAH, s=130, zorder=3, label="Terendah")
for nama, nilai in zip(BULAN, jual_juta.values):            # nilai di atas tiap titik
    ax.annotate(koma(nilai), (nama, nilai), xytext=(0, 9), textcoords="offset points",
                ha="center", fontsize=9)
ax.set_title("1. Pendapatan Bersih per Bulan, 2025")
ax.set_xlabel("Bulan")
ax.set_ylabel("Penjualan (juta Rupiah)")
ax.set_ylim(0, jual_juta.max() * 1.25)                      # sumbu mulai dari 0 agar tidak menyesatkan
ax.legend(loc="upper right")

# Kanan atas: Bagian 2, 10 produk terlaris
ax = axes[0, 1]
urut = produk.iloc[::-1]                                    # dibalik agar yang terlaris tampil di atas
warna_batang = [BIRU] * (len(urut) - 1) + [HIJAU]           # juara diberi warna lain
ax.barh(urut.index, urut["unit"], color=warna_batang)
for i, (unit, omzet) in enumerate(zip(urut["unit"], urut["omzet_juta"])):
    ax.text(unit + 1, i, f"{unit} unit | Rp {koma(omzet)} jt", va="center", fontsize=9)
ax.set_title("2. 10 Produk Terlaris, 2025 (Unit Terjual)")
ax.set_xlabel("Unit terjual")
ax.set_ylabel("Kode produk")
ax.set_xlim(0, urut["unit"].max() * 1.35)                   # ruang di kanan untuk tulisan

# Kiri bawah: Bagian 3, status transaksi
ax = axes[1, 0]
potongan, _ = ax.pie(status.values, colors=[HIJAU, MERAH, KUNING], startangle=90, counterclock=False,
                     wedgeprops={"edgecolor": "white", "linewidth": 2})
ax.text(0, -0.45, f"{koma(persen['SELESAI'])}%", ha="center", va="center",
        fontsize=16, fontweight="bold", color="white")       # persen irisan besar di dalam
label = []
for nama in status.index:
    banyak = f"{status[nama]:,}".replace(",", ".")
    label.append(f"{nama}: {banyak} transaksi ({koma(persen[nama])}%)")
ax.legend(potongan + [plt.Rectangle((0, 0), 1, 1, fc="none")], label,
          loc="upper center", bbox_to_anchor=(0.5, 0.02), frameon=False)
ax.set_title("3. Proporsi Status Transaksi")

# Kanan bawah: Bagian 4, jumlah beli vs diskon
ax = axes[1, 1]
ax.scatter(diskon_geser, jumlah_geser, s=12, color=BIRU, alpha=0.25, label="Transaksi (digeser sedikit)")
ax.plot(ringkas.index, ringkas["rata_rata"], color=MERAH, marker="o", linewidth=2,
        label="Rata-rata per tingkat diskon")
for d, r in ringkas["rata_rata"].items():                   # rata-rata dengan latar putih agar terbaca
    ax.text(d, r + 0.55, koma(r, 2), ha="center", color=MERAH, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.9))
ax.set_xticks(ringkas.index)
ax.set_xticklabels([f"{d}%\n(n={n})" for d, n in ringkas["n"].items()])   # banyak data tiap diskon
ax.set_yticks([1, 2, 3, 4, 6, 12])
ax.set_ylim(0, 16)                                          # ruang kosong di atas untuk legenda
ax.legend(loc="upper center")
ax.set_title(f"4. Jumlah Beli vs Diskon ")
ax.set_xlabel("Diskon")
ax.set_ylabel("Jumlah beli (unit per transaksi)")

fig.suptitle(f"Ringkasan Penjualan POS Tahun {TAHUN}", fontsize=16, fontweight="bold")
fig.savefig("hasil/visualisasi_gabungan_2x2.png", dpi=150, bbox_inches="tight")
plt.show()

#### Catatan sel: menggambar keempat grafik

`plt.subplots(2, 2)` membuat satu gambar berisi empat kotak. Tiap kotak dipanggil lewat `axes[baris, kolom]` dengan hitungan dimulai dari 0: `[0, 0]` kiri atas, `[0, 1]` kanan atas, `[1, 0]` kiri bawah, dan `[1, 1]` kanan bawah. `constrained_layout=True` mengatur jarak antarkotak otomatis agar judul dan label tidak saling menimpa.

Beberapa pengaturan yang disengaja:
- Sumbu Y grafik 1 dimulai dari 0 agar selisih antarbulan tidak tampak lebih besar dari aslinya.
- Grafik 2 menulis unit dan omzet di ujung batang. Produk terlaris diberi warna hijau.
- Grafik 3 memisahkan BATAL dan RETUR, dan baris legenda terakhir menjumlahkan keduanya sebagai transaksi gagal.
- Grafik 4 menulis jumlah data (n) di bawah tiap label diskon, karena rata-rata dari data sedikit kurang bisa dipercaya.
- Fungsi `koma()` menulis desimal dengan koma (16,2), sesuai kebiasaan penulisan di Indonesia.
- Gambar disimpan ke `hasil/visualisasi_gabungan_2x2.png`.


## Kesimpulan

| Bagian | Temuan utama |
|---|---|
| 1. Penjualan per bulan | Puncak di Maret (16,2 juta) dan lonjakan di Desember (15,5 juta). Terendah di April (10,9 juta). |
| 2. 10 produk terlaris | RMT1408 terlaris dalam unit dan omzet. Unit dan omzet tidak selalu sejalan. |
| 3. Status transaksi | 94,5% berhasil, 5,5% gagal/batal. Pembatalan lebih sering daripada retur. |
| 4. Jumlah beli vs diskon | Tidak tampak berkaitan (r = -0,013, rho = -0,013). |

**Catatan angka:** total penjualan di sini (Rp 151.478.786) berbeda dari tugas Analisis Pendapatan (Rp 135.001.100) karena di sini PPN 11% ditambahkan, jumlah negatif dijadikan positif (bukan dibuang), dan penjualan dibatasi pada tahun 2025.